# Vectorless Reasoning-Based RAG — a from-scratch walkthrough

This notebook builds a small, self-contained version of **vectorless RAG**: a retrieval-augmented generation pipeline that uses an LLM to *reason* over the natural structure of a document instead of searching a vector database of embeddings.

## Why bother?

In classic RAG you chunk your documents, embed every chunk into a high-dimensional vector, store those vectors in a database (pgvector, Pinecone, Chroma, etc.), and at query time you retrieve the top-k chunks by cosine similarity. It works, but it has well-known weaknesses:

- **Arbitrary chunk boundaries** can cut a sentence — or a table — in half.
- **Cosine similarity is opaque.** Two chunks scored 0.81 and 0.79; you can't really tell *why* one beat the other.
- **You need an embedding model, a vector DB, and a chunking strategy** — three moving parts that all have to stay in sync.

Vectorless RAG (popularized by [PageIndex](https://github.com/VectifyAI/PageIndex) and Microsoft's [writeup](https://techcommunity.microsoft.com/blog/azuredevcommunityblog/vectorless-reasoning-based-rag-a-new-approach-to-retrieval-augmented-generation/4502238)) drops all three:

| Classic vector RAG | Vectorless reasoning RAG |
|---|---|
| Chunk → embed → store in vector DB | Keep natural pages/sections |
| Cosine similarity to retrieve top-k | LLM reasons over a tree-of-contents to pick pages |
| Opaque "why was this retrieved?" | Fully traceable: the LLM names the pages it chose and why |
| Needs embedding model + vector store | Just an LLM and a PDF |

The tradeoff: **more LLM calls per query** (you're paying for reasoning instead of a pgvector lookup). It's a great fit for small/medium corpora where explainability and accuracy matter more than millisecond latency — finance, legal, compliance, research.

## What we'll build

A four-step pipeline:

1. **Load** a PDF and split it by *page* (no chunking, no embeddings).
2. **Index** — ask a cheap LLM to write one short summary per page; this becomes our "table of contents."
3. **Retrieve** — give the whole ToC to a stronger LLM and let it *reason* about which pages to read.
4. **Answer** by reading only those pages, with page-number citations.

Sample document: **[NIST AI 600-1 — Artificial Intelligence Risk Management Framework: Generative AI Profile](https://nvlpubs.nist.gov/nistpubs/ai/NIST.AI.600-1.pdf)** (U.S. NIST, public domain). It's on-topic for GenAI, heavily structured (numbered sections, suggested actions, page-aligned content), and safe to redistribute. Already included at `data/nist_ai_600-1.pdf`.


## 0. Setup

You need an Anthropic API key. The cell below will check for `ANTHROPIC_API_KEY` and, if it's missing, prompt you to paste one (using `getpass` so it doesn't get echoed or saved into the notebook).

To set it permanently before launching Jupyter:

```bash
export ANTHROPIC_API_KEY=sk-ant-...
```

…or drop it into a `.env` file in this folder — `python-dotenv` will pick it up automatically.


In [1]:
import os, json, pathlib, getpass
from dotenv import load_dotenv
load_dotenv()  # pulls ANTHROPIC_API_KEY from a .env file if present

import anthropic
from pypdf import PdfReader

PDF_PATH     = "data/nist_ai_600-1.pdf"
TOC_PATH     = "data/toc.json"         # cache so we only pay for indexing once
INDEX_MODEL  = "claude-haiku-4-5"      # cheap & fast — used once per page at index time
ANSWER_MODEL = "claude-sonnet-4-6"     # smarter — used for retrieval reasoning + final answer

# Make sure the API key is actually set BEFORE we build the client.
# anthropic.Anthropic() will silently succeed without a key and only blow up
# later when you try to make a real call — let's fail fast instead.
if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Paste your ANTHROPIC_API_KEY: ").strip()

assert os.environ.get("ANTHROPIC_API_KEY", "").startswith("sk-ant-"), \
    "ANTHROPIC_API_KEY is missing or malformed."

client = anthropic.Anthropic()
print("Anthropic SDK:", anthropic.__version__, "— ready.")


Anthropic SDK: 0.104.1 — ready.


## 1. Load the PDF, page by page

In classic RAG this is where you'd start splitting the document into ~500-token chunks with some overlap. We're not going to do that.

**The page is our atomic unit of retrieval.** It's a natural boundary that the document's author chose: a page usually ends at a section break or a paragraph break, tables sit on one page, footnotes stay with their text. We extract the text from each page with `pypdf` and keep a `(page_number, text)` pair. That's it — no embeddings, no chunking, no overlap.

The function below also drops any pages that came out completely empty (occasionally happens with scanned/image-only pages).


In [2]:
def load_pages(pdf_path: str) -> list[dict]:
    """Return a list of {'page': N, 'text': ...} dicts, one per non-empty page."""
    reader = PdfReader(pdf_path)
    pages = []
    for i, page in enumerate(reader.pages, start=1):
        text = (page.extract_text() or "").strip()
        if text:
            pages.append({"page": i, "text": text})
    return pages

pages = load_pages(PDF_PATH)
print(f"Loaded {len(pages)} non-empty pages")
print(f"\n--- Sample of what page {pages[10]['page']} looks like ---")
print(pages[10]["text"][:500], "...")


Loaded 64 non-empty pages

--- Sample of what page 11 looks like ---
7 
unethical behavior. Text-to-image models also make it easy to create images that could be used to 
promote dangerous or violent messages. Similar concerns are present for other GAI media, including 
video and audio. GAI may also produce content that recommends self-harm or criminal/illegal activities.  
Many current systems restrict model outputs to limit certain content or in response to certain prompts, 
but this approach may still produce harmful recommendations in response to other less-e ...


## 2. Build a tree-of-contents (one summary per page)

This is the **indexing** step. For each page we ask a cheap, fast model (Haiku) to produce a structured summary:

- `title` — a short label for the page (≤ 10 words)
- `summary` — 2-3 sentences describing what's *on* this specific page (topics, sections, entities)
- `keywords` — distinctive words/phrases someone might search for

Why this works: the model is *much* better at writing a faithful one-paragraph summary of a page than the embedding-based alternative is at picking the right chunk via cosine similarity. The summaries are concise enough that the **entire index for a 64-page document fits in a single prompt** — which is exactly what makes the retrieval step in the next cell possible.

We cache the result to `data/toc.json`. Indexing costs roughly one Haiku call per page (~64 calls for this PDF), so you don't want to redo it every time you restart the kernel. Delete the file to regenerate.


In [3]:
SUMMARY_PROMPT = """You are indexing a single page of a document for later retrieval.

Read the page text below and return a JSON object with these fields:
- "title":   a short (<= 10 words) title describing what this page is about
- "summary": 2-3 sentences capturing the main content; mention specific topics, sections, or entities
- "keywords": a list of 5-10 distinctive keywords or phrases someone might search for to find this page

Return ONLY the JSON, no preamble.

PAGE TEXT:
\"\"\"
{text}
\"\"\"
"""

def _extract_json(raw: str) -> dict:
    """Robustly pull the first {...} JSON object out of an LLM response."""
    raw = raw.strip()
    # Strip ```json ... ``` fences if present
    if raw.startswith("```"):
        raw = raw.strip("`")
        if raw.startswith("json"):
            raw = raw[4:]
        raw = raw.strip()
    # Fallback: slice between the first { and the last }
    if not raw.startswith("{"):
        start, end = raw.find("{"), raw.rfind("}")
        if start != -1 and end != -1:
            raw = raw[start : end + 1]
    return json.loads(raw)

def summarize_page(page: dict) -> dict:
    """Call the indexing model to produce a structured summary for one page."""
    msg = client.messages.create(
        model=INDEX_MODEL,
        max_tokens=400,
        messages=[{"role": "user", "content": SUMMARY_PROMPT.format(text=page["text"][:6000])}],
    )
    data = _extract_json(msg.content[0].text)
    data["page"] = page["page"]
    return data


Now run the indexing loop. The first time this runs against the NIST PDF it makes 64 Haiku calls (a few cents, under a minute). Subsequent runs load the cached JSON instantly.


In [4]:
def build_toc(pages: list[dict], cache_path: str) -> list[dict]:
    cache = pathlib.Path(cache_path)
    if cache.exists():
        print(f"Loading cached ToC from {cache_path}")
        return json.loads(cache.read_text())

    toc = []
    for p in pages:
        print(f"  indexing page {p['page']}/{len(pages)}...", end="\r")
        try:
            toc.append(summarize_page(p))
        except Exception as e:
            print(f"\n  page {p['page']} failed: {e}")
            toc.append({"page": p["page"], "title": "(indexing failed)", "summary": "", "keywords": []})
    cache.write_text(json.dumps(toc, indent=2))
    print(f"\nSaved ToC to {cache_path}")
    return toc

toc = build_toc(pages, TOC_PATH)
print(f"\nIndexed {len(toc)} pages. Here are the first three entries:\n")
for entry in toc[:3]:
    print(json.dumps(entry, indent=2))
    print("---")


Loading cached ToC from data/toc.json

Indexed 64 pages. Here are the first three entries:

{
  "title": "NIST AI Risk Management Framework for Generative AI",
  "summary": "This NIST publication (AI 600-1) presents a framework for managing risks associated with generative artificial intelligence systems. It is part of NIST's broader initiative on trustworthy and responsible AI, providing guidance on responsible AI development and deployment.",
  "keywords": [
    "NIST AI 600-1",
    "generative artificial intelligence",
    "AI risk management",
    "trustworthy AI",
    "responsible AI",
    "AI framework",
    "generative AI profile",
    "AI governance"
  ],
  "page": 1
}
---
{
  "title": "NIST AI Risk Management Framework for Generative AI",
  "summary": "This NIST publication (AI 600-1) from July 2024 presents a specialized risk management framework tailored for generative artificial intelligence systems. Published by the U.S. Department of Commerce's National Institute of Stand

## 3. Reasoning-based retrieval

This is the part that replaces your vector database.

We take the user's question, paste the **entire table of contents** into a prompt, and ask a stronger model (Sonnet) to pick the page numbers it thinks are most relevant — *and to explain its reasoning out loud*.

A few things to notice about this design:

- **No similarity scores, no top-k threshold.** The model picks however many pages it thinks the question needs (usually 2-8). For a narrow factual question it might pick one page; for a broad "summarize all governance recommendations" question it might pick a dozen.
- **The reasoning is visible.** You get a paragraph explaining *why* those pages were chosen, which means you can audit retrieval failures the same way you'd review a junior analyst's work — instead of squinting at embedding distances.
- **Latency and cost scale with ToC size, not corpus size of full text.** As long as your summaries-of-all-pages fit in one prompt (~comfortably 200+ pages with Sonnet's context window), retrieval is a single LLM call.

For really big corpora the trick is to make the ToC hierarchical (summarize sections, then pages within sections) — but for a 64-page document, flat is fine.


In [5]:
RETRIEVAL_PROMPT = """You are a retrieval agent for a document. Below is a table of contents where each entry corresponds to a single page.

Your job: choose the pages most likely to contain information needed to answer the user's question. Reason about which sections are relevant.

Return a JSON object with:
- "reasoning": a short paragraph explaining which sections of the document are relevant and why
- "pages": a list of page numbers to read (typically 1-5 pages; only include pages that genuinely help)

Return ONLY the JSON.

USER QUESTION:
{question}

TABLE OF CONTENTS:
{toc}
"""

def retrieve_pages(question: str, toc: list[dict], answer_model: str = ANSWER_MODEL) -> dict:
    """Ask the model which pages to read, and why."""
    # Compact ToC representation: one line per page
    toc_text = "\n".join(
        f"- p.{e['page']}: {e['title']} — {e['summary']}" for e in toc
    )
    msg = client.messages.create(
        model=answer_model,
        max_tokens=600,
        messages=[{"role": "user", "content": RETRIEVAL_PROMPT.format(question=question, toc=toc_text)}],
    )
    return _extract_json(msg.content[0].text)


## 4. Answer synthesis

Once we know which pages to read, we pull their *full text* (not the summaries — the actual content) and hand it to the model along with the original question.

Two key constraints in the answer prompt:

1. **"Use ONLY the document pages provided."** This is what makes the output grounded — the model shouldn't fall back on what it remembers about NIST from training.
2. **"Cite the page numbers you used, like (p. 12)."** This is the explainability payoff. Every claim in the answer is traceable to a specific page you can open and read yourself.

The `ask()` helper wires steps 3 and 4 together and also prints the retrieval reasoning so you can see exactly what the model decided to read before it answered.


In [6]:
ANSWER_PROMPT = """You are answering a user's question using ONLY the document pages provided below. If the pages do not contain enough information, say so honestly.

Cite the page numbers you used inline, like (p. 12). Be concise and specific.

USER QUESTION:
{question}

PAGES:
{pages}
"""

def ask(question: str, toc: list[dict], pages: list[dict], verbose: bool = True) -> str:
    """End-to-end: retrieve relevant pages, then synthesize a cited answer."""
    selection = retrieve_pages(question, toc)
    chosen = selection["pages"]
    page_lookup = {p["page"]: p["text"] for p in pages}
    pages_text = "\n\n".join(
        f"=== Page {n} ===\n{page_lookup.get(n, '(page not found)')}" for n in chosen
    )

    if verbose:
        print("Retrieval reasoning:")
        print(" ", selection["reasoning"])
        print("Chosen pages:", chosen)
        print()

    msg = client.messages.create(
        model=ANSWER_MODEL,
        max_tokens=800,
        messages=[{"role": "user", "content": ANSWER_PROMPT.format(question=question, pages=pages_text)}],
    )
    return msg.content[0].text.strip()


## 5. Try it

Three queries against the NIST GenAI Profile, ranging from broad ("what governance actions?") to narrow ("define this one term"). For each one, watch the **retrieval reasoning** that prints first — that's the model thinking out loud about which pages to look at — and then the cited answer.

This is the demo to show people: ask any question, and you can audit both *what the model retrieved* and *why*, then verify every citation against the source PDF.


In [7]:
question = "What are some suggested actions for governing risks specific to generative AI?"
print(ask(question, toc, pages))


Retrieval reasoning:
  The user is asking about suggested actions for governing risks specific to generative AI. The most relevant pages would be those that outline specific governance actions, policies, and procedures within the AI RMF framework for GAI. Pages 17-26 cover the GOVERN function with specific suggested actions (GV subcategories), and page 16 introduces the structure of suggested actions. Pages 18, 19, 22, 23, 24, and 25 specifically outline governance policies, risk oversight, feedback mechanisms, and third-party risk management with concrete suggested actions. Page 4 and 5 also provide a high-level overview of the framework's purpose regarding suggested actions for GAI risks.
Chosen pages: [16, 17, 18, 19, 22, 23]



## Suggested Actions for Governing Generative AI Risks

The document outlines numerous suggested actions organized by AI Risk Management Framework (AI RMF) subcategories. Here are the key governance actions:

### Legal & Regulatory Compliance
- Align GAI development with applicable laws on data privacy, copyright, and intellectual property (p. 17)

### Transparency & Policies
- Establish transparency policies documenting the origin and history of training and generated data (p. 18)
- Establish policies to evaluate risk-relevant capabilities and robustness of safety measures before and after deployment (p. 18)

### Risk Tiering & Thresholds
- Update risk tiers considering factors like information integrity, harm to fundamental rights, psychological impacts, and malicious use potential (p. 18)
- Establish minimum performance/assurance thresholds for deployment approval ("go/no-go" policies) (p. 18)
- Devise a plan to **halt development or deployment** of GAI systems posing unacceptable r

In [8]:
question = "Which risks does NIST identify as unique to or exacerbated by generative AI compared to traditional AI?"
print(ask(question, toc, pages))


Retrieval reasoning:
  The user is asking specifically about risks that NIST identifies as unique to or exacerbated by generative AI compared to traditional AI. The most relevant pages are those that directly describe and categorize these GAI-specific risks. Page 4 provides a comprehensive overview of risks unique to or exacerbated by GAI. Page 5 explicitly mentions 'risks novel to or exacerbated by GAI technologies.' Pages 8 and 9 outline the nine major risk categories for GAI systems. Pages 10-15 go into detail on specific risk categories like confabulation, harmful content, data privacy, environmental impact, bias, human-AI configuration, information integrity, and cybersecurity. Page 7 discusses dimensions and characteristics of GAI risks. These pages together cover the full scope of NIST's identified GAI-specific risks.
Chosen pages: [4, 5, 7, 8, 9]



Based on the document pages provided, NIST identifies **12 risks unique to or exacerbated by generative AI (GAI)** (p. 8-9):

1. **CBRN Information or Capabilities** – Eased access to information related to chemical, biological, radiological, or nuclear weapons (p. 8)

2. **Confabulation** – Production of confidently stated but false content ("hallucinations") that can mislead users (p. 8)

3. **Dangerous, Violent, or Hateful Content** – Eased production of violent, radicalizing, or self-harm-encouraging content (p. 8)

4. **Data Privacy** – Leakage or unauthorized use of personally identifiable or sensitive information (p. 8)

5. **Environmental Impacts** – High compute resource utilization in training/operating GAI models adversely affecting ecosystems (p. 8)

6. **Harmful Bias or Homogenization** – Amplification of historical/societal biases and undesired homogeneity in outputs (p. 8)

7. **Human-AI Configuration** – Inappropriate anthropomorphization, over-reliance, automation bias

In [9]:
question = "How does the document define 'confabulation' and why is it a risk?"
print(ask(question, toc, pages))


Retrieval reasoning:
  The user is asking about how the document defines 'confabulation' and why it is a risk. Page 10 is explicitly dedicated to 'Confabulation in Generative AI Systems' and directly defines the term and discusses its risks. Page 8 lists confabulation as one of nine major risk categories, providing a high-level overview. Page 39 mentions confabulation among other risks in the context of AI model explanation and documentation measures. These three pages are most likely to contain the relevant definition and risk explanation.
Chosen pages: [10, 8, 39]



## Definition of 'Confabulation'

The document defines **confabulation** as "a phenomenon in which GAI systems generate and confidently present erroneous or false content in response to prompts," including outputs that diverge from prompts or contradict previously generated statements in the same context (p. 10). It is also colloquially known as "hallucinations" or "fabrications" (p. 8, p. 10).

### Why It Occurs

Confabulation is described as **a natural byproduct of how generative models work**: they generate outputs approximating the statistical distribution of their training data (e.g., LLMs predicting the next word/token), which can produce factually inaccurate or internally inconsistent results — especially with open-ended prompts or domain-specific questions (p. 10).

### Why It Is a Risk

Several risks are identified (p. 10):

- **User deception**: Users may believe false content due to the **confident tone** of the response and act upon or promote it.
- **High-stakes domains**

## Where to go next

Now that the core loop works, the natural extensions are:

- **Hierarchical ToC.** For documents longer than a few hundred pages, summarize *sections* (groups of pages) on top of page summaries. Retrieval then walks the tree top-down — first pick the section, then the pages within it — instead of scanning every page summary in one call.
- **Multi-document corpora.** Add a per-document description and a routing step that picks the right document before picking pages. PageIndex calls this the "library" layer.
- **Prompt caching.** The ToC is reused across every query — wrap it in an Anthropic [`cache_control`](https://docs.anthropic.com/en/docs/build-with-claude/prompt-caching) block to drop retrieval cost by ~90%.
- **Compare against vector RAG.** Run both pipelines on the same questions and look at: answer quality, latency, cost per query, and (the most useful metric) how often you can trace *why* a given chunk was retrieved. That's where vectorless usually wins.
- **Swap in your own document.** Change `PDF_PATH`, delete `data/toc.json`, and re-run. Anything with clear section structure (financial filings, legal contracts, technical specs, research papers) is a good candidate.
